# Isaac Sim Jupyter Notebook Examples

This notebook will demonstrate step by step how to load the IAI apartment into the simulation environment and import a Anymal robot.

> The VNC desktop needs to be opened (Run the launcher.ipynb).
> 
> "Control + Enter" to execute the selected code cell. 

<!-- <button data-commandlinker-command="notebook:restart" class="jupyter-button">Force Stop</button> -->

## Start Main App

> Note: The application window is frozen and non-interactive, which is normal.

In [ ]:
from isaacsim import SimulationApp

simulation_app = SimulationApp({
    "headless": False,
    # "hide_ui": True,
    "width": 1280,
    "height": 960,
    "renderer": "RaytracedLighting",
    "display_options": 3286,  # Set display options to show default grid
})

## Import Libraries

In [ ]:
import carb
import os
import numpy as np
from isaacsim.core.api import World
from isaacsim.core.utils.prims import define_prim
from isaacsim.core.utils import viewports
from isaacsim.robot.policy.examples.robots import AnymalFlatTerrainPolicy

## Define the physical properties of the simulation environment

In [ ]:
my_world = World(stage_units_in_meters=1.0, physics_dt=1 / 200, rendering_dt=8 / 200)

## Spawn USD to the world

The apartment USD is directly converted from [iai apartment URDF](https://github.com/code-iai/iai_maps/blob/ros-jazzy/iai_apartment/urdf/apartment.urdf).

In [ ]:
prim = define_prim("/World/apartment", "Xform")
asset_path = f"{os.getcwd()}/../usd/apartment/apartment.usd"
prim.GetReferences().AddReference(asset_path)

## Refresh View

Until now, we don't see any change in the app window, that's because it needs to be manually refreshed.

In [ ]:
def refresh_view(steps = 10):
    for i in range(steps):
        my_world.step(render=True)

refresh_view()

## Change camera position

In [ ]:
viewports.set_camera_view(eye=np.array([5.5, 0, 3]), target=np.array([0, 0, 0]))
refresh_view()

## Spawn Robot Anymal

The USD file for the Anymal robot is provided by Isaac Sim itself.

In [ ]:
robot = AnymalFlatTerrainPolicy(
    prim_path="/World/Anymal",
    name="Anymal",
    position=np.array([0, 0, 0.5]),
)
refresh_view()

## Start the physics simulation

You will see the robot collapse on the ground because no control commands have been sent to it yet.

In [ ]:
my_world.reset()
refresh_view(steps=50)

## Add physics callback function to control the robot

The command is a 3-element array, where the first value represents forward velocity, the second represents lateral (left/right) movement, and the third represents rotation. Value range is from -1 to 1.

In [ ]:
first_step = True
commands = [0.0, 0.0, 0.0]

def on_physics_step(step_size) -> None:
    global first_step, commands
    if first_step:
        robot.initialize()
        first_step = False
    else:
        robot.forward(step_size, commands)

my_world.add_physics_callback("physics_step", callback_fn=on_physics_step)

### Restart Simulation and initialize robot

In [ ]:
first_step = True
my_world.reset()
refresh_view(steps=50)

### Send Control Commands

In [ ]:
# Forward
commands = [0.3, 0.0, 0.0]
refresh_view(steps=200)
# Move left
commands = [0.0, 0.3, 0.0]
refresh_view(steps=200)
# Turn around
commands = [0.0, 0.0, 0.8]
refresh_view(steps=200)

## Running the simulation continuously

Once the following code cell is executed, it will enter an infinite loop. You can only terminate the entire program by clicking the "Shutdown" button below, after which you'll need to rerun the previous code.

<button data-commandlinker-command="notebook:restart-clear-output" class="jupyter-button">Shutdown</button>

In [ ]:
first_step = True
my_world.reset()

plans = [
    [0.3, 0.0, 0.0],
    [0.0, 0.0, 0.8],
    [0.3, 0.0, 0.0],
    [-0.3, 0.0, 0.0],
    [0.0, 0.0, 0.0],
]

commands = plans.pop(0)

N = 100
frame_count = 0

# while len(plans) != 0:
while simulation_app.is_running():
    my_world.step(render=True)
    # change command every 30 frames
    if my_world.is_playing() and len(plans) != 0:
        frame_count += 1
        if frame_count % N == 0:
            commands = plans.pop(0)
            print("Active command:", commands)